In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/UDB/Riset/2026")

CHECKPOINTS_DIR = PROJECT_ROOT / "results/checkpoints"
FEATURE_DIR = CHECKPOINTS_DIR / "features"

required_files = [
    CHECKPOINTS_DIR / "dataset_metadata.parquet",
    CHECKPOINTS_DIR / "resolved_columns.json",
    CHECKPOINTS_DIR / "experiment_01_outputs.joblib",
    CHECKPOINTS_DIR / "HMRNet-Full.pt",
    FEATURE_DIR / "semantic_embeddings.npy",
]

for file_path in required_files:
    print(f"{file_path.name}: {file_path.exists()}")

# 05 — Downstream Task Utility

Opsi default adalah sentiment classification. Topic embedding ditambahkan
ke text embedding dan dibandingkan dengan Transformer-only, LDA,
BERTopic, Top2Vec, serta HMRNet.


In [ ]:
from pathlib import Path
import sys

# Sesuaikan hanya apabila folder proyek berada di lokasi lain.
PROJECT_ROOT = Path("/content/drive/MyDrive/UDB/Riset/2026")
if not PROJECT_ROOT.exists():
    DRIVE_ROOT = Path("/content/drive/MyDrive/HMRNet_Experiments")
    if DRIVE_ROOT.exists():
        PROJECT_ROOT = DRIVE_ROOT

assert PROJECT_ROOT.exists(), (
    "Folder HMRNet_Experiments tidak ditemukan. "
    "Unggah/ekstrak proyek ke /content atau Google Drive."
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"
LOGS_DIR = RESULTS_DIR / "logs"
FEATURE_DIR = CHECKPOINTS_DIR / "features"

for directory in [TABLES_DIR, FIGURES_DIR, CHECKPOINTS_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
%cd {PROJECT_ROOT}

%pip install -q \
    scikit-learn \
    pandas \
    numpy \
    joblib \
    openpyxl \
    pyarrow

In [ ]:
import numpy as np
import pandas as pd
from src.preprocessing import load_feature_bundle

bundle = load_feature_bundle(FEATURE_DIR)
raw_texts = bundle["raw_texts"]
processed_texts = bundle["processed_texts"]
bow = bundle["bow"]
bow_normalized = bundle["bow_normalized"]
tfidf = bundle["tfidf"]
semantic_embeddings = bundle["semantic_embeddings"]
document_embeddings = bundle["document_embeddings"]
vocabulary = bundle["vocabulary"]

print("Documents:", len(raw_texts))
print("Vocabulary:", len(vocabulary))
print("Semantic shape:", semantic_embeddings.shape)
print("Document shape:", document_embeddings.shape)


In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import make_pipeline

from src.hmrnet import HMRNet

metadata_df = pd.read_parquet(
    CHECKPOINTS_DIR / "dataset_metadata.parquet"
)
columns = json.loads(
    (CHECKPOINTS_DIR / "resolved_columns.json").read_text()
)

if not columns.get("label_column"):
    raise ValueError(
        "Dataset tidak memiliki kolom sentiment/label. "
        "Tetapkan LABEL_COLUMN pada notebook 00."
    )

y_raw = metadata_df[columns["label_column"]].astype(str).to_numpy()
if len(y_raw) != len(raw_texts):
    raise ValueError(
        "Jumlah label tidak sama dengan feature bundle. "
        "Jalankan ulang notebook 00 tanpa mengubah urutan data."
    )


In [ ]:
encoder = LabelEncoder()
y = encoder.fit_transform(y_raw)

indices = np.arange(len(y))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

baseline_outputs = joblib.load(
    CHECKPOINTS_DIR / "experiment_01_outputs.joblib"
)
hmrnet = HMRNet.load(
    CHECKPOINTS_DIR / "HMRNet-Full.pt",
    device="cuda" if torch.cuda.is_available() else "cpu",
)
hmrnet_topic = hmrnet.transform(
    bow, semantic_embeddings, document_embeddings
)

print("Classes:", list(encoder.classes_))


In [ ]:
representations = {
    "Transformer only": semantic_embeddings,
    "Transformer + HMRNet": np.hstack(
        [semantic_embeddings, hmrnet_topic]
    ),
}

for name in ["LDA", "BERTopic", "Top2Vec"]:
    output = baseline_outputs.get(name)
    if output is None:
        print(f"{name} tidak tersedia; dilewati.")
        continue
    topic_vector = output.get("document_topic")
    if topic_vector is None or len(topic_vector) != len(y):
        print(f"{name} document-topic tidak valid; dilewati.")
        continue
    representations[f"Transformer + {name}"] = np.hstack(
        [semantic_embeddings, topic_vector]
    )


In [ ]:
rows = []
probabilities_by_model = {}

for name, features in representations.items():
    classifier = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=42,
        ),
    )
    classifier.fit(features[train_idx], y[train_idx])
    predictions = classifier.predict(features[test_idx])
    probabilities = classifier.predict_proba(features[test_idx])
    probabilities_by_model[name] = probabilities

    row = {
        "Model": name,
        "Accuracy": accuracy_score(y[test_idx], predictions),
        "Macro_F1": f1_score(
            y[test_idx], predictions, average="macro"
        ),
        "Weighted_F1": f1_score(
            y[test_idx], predictions, average="weighted"
        ),
    }

    try:
        row["ROC_AUC"] = roc_auc_score(
            y[test_idx],
            probabilities,
            multi_class="ovr",
            average="macro",
        )
    except ValueError:
        row["ROC_AUC"] = np.nan

    rows.append(row)

downstream_results = pd.DataFrame(rows).sort_values(
    "Macro_F1", ascending=False
)
downstream_results.to_csv(
    TABLES_DIR / "experiment_05_sentiment_classification.csv",
    index=False,
)
downstream_results.to_excel(
    TABLES_DIR / "experiment_05_sentiment_classification.xlsx",
    index=False,
)
display(downstream_results.round(4))


In [ ]:
import matplotlib.pyplot as plt

plot_frame = downstream_results.sort_values("Macro_F1")
plt.figure(figsize=(9, 5))
plt.barh(plot_frame["Model"], plot_frame["Macro_F1"])
plt.xlabel("Macro-F1")
plt.title("Downstream Sentiment Classification Utility")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "experiment_05_macro_f1.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/UDB/Riset/2026")

TABLES_DIR = PROJECT_ROOT / "results/tables"
FIGURES_DIR = PROJECT_ROOT / "results/figures"
CHECKPOINTS_DIR = PROJECT_ROOT / "results/checkpoints"
FEATURE_DIR = CHECKPOINTS_DIR / "features"

required_files = [
    TABLES_DIR / "experiment_05_sentiment_classification.csv",
    CHECKPOINTS_DIR / "HMRNet-Full.pt",
    FEATURE_DIR / "semantic_embeddings.npy",
]

for file_path in required_files:
    print(f"{file_path.name}: {file_path.exists()}")